In [1]:
import inspect
import os
from pathlib import Path

os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"

import torch
from contextlib import contextmanager
import torch._higher_order_ops.utils as _hou

# torch 2.11 半成品缺这个符号，transformers 一加载就会炸
if not hasattr(_hou, "setup_compilation_env"):
    @contextmanager
    def _setup_compilation_env(*_args, **_kwargs):
        yield
    _hou.setup_compilation_env = _setup_compilation_env

from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

# 1. 路径：按云端实际位置改。本机一般是 cleaned_data/qa/
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
TRAIN_FILE = "../mix_train.jsonl"
VAL_FILE = "../val.jsonl"
TRANSLATE_VAL_FILE = "../yue_val.jsonl"  # 没有就只用问答 val
OUTPUT_DIR = "outputs_yue_qwen"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=3407,
)

dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "eval": VAL_FILE})

SYSTEM = (
    "你是一个粤语助手。用粤语回答问题、完成写作或对话。"
    "只有用户明确要求翻译时才翻译。不要把普通提问当成翻译任务。"
)

def formatting_prompts_func(examples):
    if "text" in examples:
        return {"text": examples["text"]}
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        user = f"{inst}\n\n{inp}" if inp else inst
        texts.append(
            f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
            f"<|im_start|>user\n{user}<|im_end|>\n"
            f"<|im_start|>assistant\n{out}<|im_end|>"
        )
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True, num_proc=1)

eval_dataset = dataset["eval"]
best_metric = "eval_loss"
if Path(TRANSLATE_VAL_FILE).exists():
    tr_val = load_dataset("json", data_files=TRANSLATE_VAL_FILE, split="train")
    tr_val = tr_val.shuffle(seed=42).select(range(min(1000, len(tr_val))))
    tr_val = tr_val.map(formatting_prompts_func, batched=True, num_proc=1)
    eval_dataset = {"qa": dataset["eval"], "translate": tr_val}
    best_metric = "eval_qa_loss"
    print(f"验证：问答 {len(dataset['eval'])} + 翻译 {len(tr_val)}")
else:
    print(f"验证：仅问答 {len(dataset['eval'])}（未找到 {TRANSLATE_VAL_FILE}）")

# 5 万条左右，总 batch=16，3000 step 大约 1 个 epoch 量级（packing 会更多）
train_args = dict(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=150,
    max_steps=3000,
    learning_rate=5e-5,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model=best_metric,
    greater_is_better=False,
    report_to="none",
)

sft_kw = {
    "model": model,
    "train_dataset": dataset["train"],
    "eval_dataset": eval_dataset,
}
sig = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in sig:
    sft_kw["processing_class"] = tokenizer
else:
    sft_kw["tokenizer"] = tokenizer

try:
    from trl import SFTConfig
    sft_kw["args"] = SFTConfig(
        packing=True,
        eval_packing=False,
        dataset_text_field="text",
        max_seq_length=2048,
        dataset_num_proc=1,
        **train_args,
    )
    trainer = SFTTrainer(**sft_kw)
except TypeError:
    if "dataset_text_field" in sig:
        sft_kw["dataset_text_field"] = "text"
        sft_kw["max_seq_length"] = 2048
        sft_kw["dataset_num_proc"] = 1
        sft_kw["packing"] = True
        if "eval_packing" in sig:
            sft_kw["eval_packing"] = False
    sft_kw["args"] = TrainingArguments(**train_args)
    trainer = SFTTrainer(**sft_kw)

print("粤语助手 LoRA：从底模重新训练（不 resume 旧 checkpoint）")
trainer_stats = trainer.train(resume_from_checkpoint=False)

model.save_pretrained("yue_qwen_lora")
tokenizer.save_pretrained("yue_qwen_lora")
print("训练完成，LoRA 已保存至 yue_qwen_lora")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/miniconda3/envs/py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/usr/local/miniconda3/envs/py310/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /usr/local/miniconda3/envs/py310/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.558 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 339/339 [00:01<00:00, 291.30it/s]


unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


...

🔥 粤语翻译模型微调正式启动...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,130,950 | Num Epochs = 1 | Total steps = 10,000
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
9000,1.387751,1.392519
10000,1.445427,1.390062


/usr/local/miniconda3/envs/py310/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/miniconda3/envs/py310/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/miniconda3/envs/py310/lib/python3.10/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated 

✅ 训练完成，模型已保存至 yue_qwen_lora
